# BWE Stufe 1 — Regressions-Training (Kaggle/GPU)

**Auf Kaggle ausführen.** Im rechten Settings-Panel des Notebooks:

1. **Accelerator → GPU** (z. B. T4 ×2 oder P100). Ist die Auswahl ausgegraut oder bleibt
   die GPU-Prüfung unten bei `GPUs: []`: meist fehlt die **Telefon-Verifizierung**
   (Profilbild → *Settings* → *Phone verification*) — sie schaltet GPU **und** Internet
   frei. Sonst: Wochen-Kontingent (~30 GPU-h) erschöpft, oder laufende Session erst
   stoppen, dann Accelerator umstellen.
2. **Internet → On** (Schalter im rechten Panel, zwischen *Environment* und *Tags*;
   braucht ebenfalls Telefon-Verifizierung). Ohne Internet scheitert `git clone`
   („Could not resolve host: github.com").
3. **Add Data → MUSDB18-HQ** einbinden (z. B. `quanglvitlm/musdb18-hq`) und in Zelle 2
   `BWE_DATA_ROOT` auf den Ordner zeigen lassen, der `train/` und `test/` enthält
   (exakten Pfad im *Input*-Panel ablesen; ein evtl. `val/`-Ordner wird ignoriert).

Repo ist public → Clone ohne Token. Lokal nicht ausführbar (klont Repo, braucht GPU + Datensatz).

In [ ]:
# GPU prüfen
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Code holen & installieren

TF ist auf Kaggle vorinstalliert → `--no-deps`, damit die GPU-Build nicht ersetzt wird; nur die Audio-Pakete nachinstallieren.

In [ ]:
REPO = "https://github.com/DanyelRychter/bwe_dnn.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo

import subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/bwe_dnn"], check=True)
%pip install -q -e /kaggle/working/bwe_dnn --no-deps
%pip install -q librosa soundfile soxr

## 2. Pfade setzen (VOR dem bwe-Import!)

`BWE_DATA_ROOT` = Ordner, der `train/` und `test/` enthält. Ein evtl. vorhandener `val/`-Ordner wird ignoriert (kanonischer Split via Track-Namen).

In [ ]:
import os
# >>> An die tatsächliche Dataset-Struktur anpassen: <<<
os.environ["BWE_DATA_ROOT"] = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_CKPT_ROOT"] = "/kaggle/working/bwe_runs"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys; sys.path.insert(0, "/kaggle/working/bwe_dnn")
from bwe import config as cfg
print(cfg.summary())

In [ ]:
# Datensatz-Check: erwartete 86/14/50 (bzw. tatsächliche Zahlen)
from bwe.data import splits as SP
for s in SP.SPLIT_NAMES:
    print(f"{s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 2b. 32-kHz-Cache (einmalig pro Session, empfohlen fürs Volltraining)

Im Subset-Lauf war die **GPU nur ~17 % ausgelastet**: die `tf.data`-Pipeline resampelt jeden
Crop on-the-fly von 44,1 kHz auf 32 kHz auf der CPU → das Training ist I/O-/CPU-Bound. Diese
Zelle resampelt die 4 Stems **aller** `train/`- und `test/`-Tracks **einmal** nach
`/kaggle/working/musdb18hq_32k` und biegt anschließend `BWE_DATA_ROOT` (genauer: `cfg.DATA_ROOT`)
darauf. Danach gilt im Training `sr == cfg.SR`, der Resample-Zweig entfällt → **GPU-Bound**.

- **Stereo bleibt erhalten** → die Kanal-Augmentation (`mean`/`left`/`right`) funktioniert
  unverändert; Trainings-Semantik identisch, nur ohne den Resample-Overhead.
- **`mixture.wav` wird nicht gecacht** (Mixture = Summe der Stems) → spart Platz und Zeit.
- **Resumebar:** bereits gecachte Dateien werden übersprungen; bei Session-Neustart
  (`/kaggle/working` ist flüchtig) einfach erneut ausführen.
- Läuft einmalig je nach Tracks/IO einige Minuten. Die `train(...)`-Zellen unten bleiben
  unverändert — sie lesen automatisch aus dem umgebogenen `cfg.DATA_ROOT`.
- Diese Zelle **nach** der Pfad-/Datensatz-Check-Zelle und **vor** dem Training ausführen.

In [ ]:
# # 32-kHz-Cache: einmal alle Stems resampeln, dann BWE_DATA_ROOT darauf zeigen.
# # (Beim Subset-Lauf lief die GPU nur ~17 % — die Pipeline resampelt sonst jeden Crop
# #  on-the-fly 44,1k->32k auf der CPU = I/O-/CPU-Bound. Hier zahlen wir das EINMAL,
# #  danach gilt sr==cfg.SR und der Resample-Zweig der Pipeline entfaellt -> GPU-Bound.)
# import os, time
# from pathlib import Path
# import soundfile as sf
# import soxr
# from bwe import config as cfg
# from bwe.data import splits as SP

# CACHE_ROOT = Path("/kaggle/working/musdb18hq_32k")   # Session-fluechtig; pro Lauf neu

# # get_split liest aktuell noch aus der Quelle (cfg.DATA_ROOT zeigt auf den Kaggle-Input).
# # valid-Tracks liegen physisch im train/-Ordner -> track.subset spiegelt das korrekt,
# # sodass der Cache dieselbe Ordnerstruktur hat und der 86/14/50-Split reproduziert wird.
# all_tracks = SP.get_split("train") + SP.get_split("valid") + SP.get_split("test")
# print(f"Cache-Ziel: {CACHE_ROOT}  |  {len(all_tracks)} Tracks x {len(cfg.STEMS)} Stems")
# print("Stereo bleibt erhalten (Kanal-Augmentation mean/left/right). mixture wird NICHT")
# print("gecacht (wird aus den Stems summiert) -> spart Platz & Zeit.\n")

# t0 = time.time()
# for i, track in enumerate(all_tracks, 1):
#     out_dir = CACHE_ROOT / track.subset / track.name
#     out_dir.mkdir(parents=True, exist_ok=True)
#     for stem in cfg.STEMS:
#         dst = out_dir / f"{stem}.wav"
#         if dst.exists():                       # resumebar: schon gecacht -> ueberspringen
#             continue
#         info = sf.info(str(track.stems[stem]))
#         data, sr = sf.read(str(track.stems[stem]), always_2d=True, dtype="float32")
#         if sr != cfg.SR:
#             data = soxr.resample(data, sr, cfg.SR, quality="HQ")   # == soxr_hq der Pipeline
#         sf.write(str(dst), data, cfg.SR, subtype=info.subtype or "PCM_16")
#     if i % 10 == 0 or i == len(all_tracks):
#         print(f"  [{i:3d}/{len(all_tracks)}] {time.time()-t0:6.0f}s  {track.name[:42]}")

# # BWE_DATA_ROOT auf den Cache umbiegen. WICHTIG: cfg.DATA_ROOT direkt setzen --
# # os.environ allein wirkt nicht mehr, weil cfg den Wert beim Import gelesen hat.
# os.environ["BWE_DATA_ROOT"] = str(CACHE_ROOT)
# cfg.DATA_ROOT = CACHE_ROOT
# print(f"\nFertig in {time.time()-t0:.0f}s. cfg.DATA_ROOT -> {cfg.DATA_ROOT}")
# for s in SP.SPLIT_NAMES:                        # Gegencheck aus dem Cache: erwartet 86/14/50
#     print(f"  {s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 3. Subset-Training (Architektur/Hyperparameter)

Wenige Tracks → schnell prüfen, dass Val-LSD-HF unter die Copy-Up-Baseline sinkt.

In [ ]:
from bwe.train.regression import train
model, hist = train(run="reg_subset", mode="subset", batch_size=16, epochs=30, limit=20)

## 4. Volltraining

Voller Datensatz, resumebar (BackupAndRestore). Bricht eine Session ab (12-h-Grenze), Zelle einfach erneut ausführen.

In [ ]:
from bwe.train.regression import train
model, hist = train(run="reg_full", mode="full", batch_size=16, epochs=cfg.EPOCHS)

## 5. Lernkurve + Persistenz

Gewichte unter `/kaggle/working/bwe_runs/reg_full/` (best.weights.h5 + generator.weights.h5).
Für Phase D / das Ergebnis-Notebook herunterladen oder als Kaggle-Output-Dataset sichern.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("/kaggle/working/bwe_runs/reg_full/log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log["epoch"], log["loss"], label="train"); ax[0].plot(log["epoch"], log["val_loss"], label="val")
ax[0].set_title("RI+Mag-Loss"); ax[0].set_xlabel("Epoche"); ax[0].legend()
ax[1].plot(log["epoch"], log["lsd_hf"], label="train"); ax[1].plot(log["epoch"], log["val_lsd_hf"], label="val")
ax[1].set_title("LSD-HF [dB]"); ax[1].set_xlabel("Epoche"); ax[1].legend()
plt.tight_layout(); plt.show()